# SPK-10 — (Deep) Spark internals sampler

**Show → Read → Note.** Not a single pathology like SPK-1…SPK-9 — a short, demo-driven
tour of the machinery *underneath* the failures you've already broken: how **Catalyst** rewrites
your query, how **Tungsten** turns a stage into JVM bytecode, how memory splits between execution
and storage, and three cluster-launch knobs (serialization, speculation, broadcast) you tune but
rarely *see*.

**Pre-requisite:** the unified Spark server is up (`make up`); the default **tuned** box is fine.
This notebook connects via **Spark Connect**. Open the Spark UI at http://localhost:4040 if you want
the **SQL / DataFrame** plan rendered — but the workhorse here is `df.explain(...)` printed inline.

**Laptop-safe:** this module is about **plans, not volume** — data is generated lazily and only
`count()`-ed (never collected or written). Nothing to delete at the end.

> ### The one framing that matters: Connect shows the *plan*, the JVM owns the *internals*
> - ✅ **Observable over Connect** — anything in the **query plan**: `df.explain(extended=True)`
>   (Catalyst's four plans), `df.explain(mode="formatted")` / `df.explain("codegen")` (Tungsten
>   codegen), and any SQL conf via `spark.conf.get(...)`. These `explain()` calls **are the artifact.**
> - ❌ **NOT observable over Connect** — `spark.sparkContext`, **RDDs**, **accumulators**, and
>   broadcast *variables* (`sc.broadcast`). They need the JVM driver and **raise on Connect**. Where a
>   concept only lives there, we **describe it and show the DataFrame-level analogue.**

See the companion writeup in [`README.md`](./README.md) and the
[Spark-UI guide](../../docs/spark-ui-guide.md).

In [ ]:
from common.spark_session import spark, display_df
from common.profiles import apply_profile, profile_summary
from common.datagen import uniform_keys, skewed_keys, key_dimension
from common.metrics_diff import measure, compare
from pyspark.sql import functions as F

# Start on the production-tuned profile (AQE on, broadcast enabled).
apply_profile(spark, "tuned")
print("\nSpark UI:", "http://localhost:4040")
spark

## Step 0 — A tiny fact + dimension to plan over

We synthesize a small `orders` fact and a `customers` dimension — lazily, nothing stored. This
module reads **plans**, not data, so the row count is deliberately modest; everything we run is a
`count()` or an `explain()`.

In [ ]:
N_ROWS = 2_000_000          # small on purpose — this module is about plans, not volume
N_KEYS = 2_000

fact = uniform_keys(spark, n_rows=N_ROWS, n_keys=N_KEYS, key_col="customer_id")
dim  = key_dimension(spark, n_keys=N_KEYS, key_col="customer_id")

print("fact columns:", fact.columns, "  dim columns:", dim.columns)
print(f"fact: ~{N_ROWS:,} rows (lazy)   dim: {N_KEYS:,} customers (small -> broadcastable)")

## (1) WholeStageCodegen — Tungsten compiles a whole stage to bytecode  ✅ *observable*

A chained query (`filter → withColumn → groupBy → agg`). `explain(mode="formatted")` shows the
operator tree (fused operators carry a `*` under a `WholeStageCodegen` node); `explain("codegen")`
shows the **generated Java** Tungsten compiles to bytecode at runtime.

In [ ]:
chained = (fact
    .filter(F.col("amount") > 50.0)
    .withColumn("amount_2x", F.col("amount") * 2)
    .groupBy("customer_id")
    .agg(F.sum("amount_2x").alias("total")))

chained.explain(mode="formatted")          # tree: look for `*` and `WholeStageCodegen (N)`
print("\n================ generated code (skim for `processNext` / one tight loop) ================\n")
chained.explain("codegen")                  # the actual Java fused into a single row loop

**What you just saw.** Adjacent operators collapsed into one **`WholeStageCodegen`** node (each
marked `*`), and the *Generated code* is real Java fused into a single loop over the rows. Instead of
an interpreter calling `next()` per row per operator (the classic *Volcano* model), Tungsten emits
one tight loop per stage — fewer virtual calls, better CPU/cache behavior. The `*` is your
at-a-glance "this stage is codegen'd" marker, and it's **fully visible over Connect** because it
lives in the physical plan.

## (2) Catalyst optimizer — the four plans, and two rules in action  ✅ *observable*

`explain(extended=True)` prints **Parsed → Analyzed → Optimized → Physical**. Watch two rules with
the naked eye: **constant folding** (`lit(2) * lit(3)` → `6`) and **column pruning** (the
unreferenced `key_label` disappears before the scan).

In [ ]:
catalyst_demo = (dim
    .select("customer_id", "tier")             # never select key_label -> should be pruned
    .withColumn("const", F.lit(2) * F.lit(3))   # constant -> should fold to 6
    .filter(F.col("customer_id") < 100))

catalyst_demo.explain(extended=True)

**What you just saw.** Compare **Analyzed** to **Optimized**: `(2 * 3)` is already reduced to the
literal `6` (**constant folding** — not recomputed per row), and `key_label` is dropped before the
scan (**column pruning / pushed projection**). Catalyst is a **rule-based + cost-based** rewriter:
your DataFrame/SQL is a *declaration*; the **Optimized** plan is what Spark intends to run. That
Analyzed→Optimized gap is the optimizer made visible — and it's **readable over Connect**. (It's also
*why* a `CAST`/UDF in a filter can block partition pruning in SPK-7: it stops a rule from firing.)

## (3) Tungsten / unified memory model  ⚠️ *confs yes, live split no (JVM)*

No JVM poking — we **read the confs** and explain the model. Execution memory (shuffle/sort/agg
buffers) and storage memory (cache) share **one** unified region and borrow from each other.

In [ ]:
for k in ["spark.memory.fraction",
          "spark.memory.storageFraction",
          "spark.memory.offHeap.enabled",
          "spark.memory.offHeap.size"]:
    print(f"{k:<35} = {spark.conf.get(k, '<unset / JVM default>')}")

**What you just saw.** `spark.memory.fraction` (~0.6) is the **unified** region used by **both**
execution and storage; `spark.memory.storageFraction` (~0.5) is the slice cache is *guaranteed*
before it must yield. The key asymmetry: **execution can evict cached blocks, but cache cannot evict
execution buffers** — the model behind **SPK-2** (executor OOM) and **SPK-8** (cache thrash).
Off-heap (`spark.memory.offHeap.*`) moves buffers outside the JVM heap to dodge GC. The conf values
read fine over Connect, but the **live execution-vs-storage split is a JVM/Executors-tab thing.**

## (4) Serialization — Kryo vs Java  ⚠️ *conf yes; it's a launch setting*

We **read** the active serializer. It matters when bytes move or rest: **shuffle** and **cached**
(`MEMORY_*_SER`) data.

In [ ]:
print("spark.serializer                =", spark.conf.get("spark.serializer", "<JVM default: JavaSerializer>"))
print("spark.kryo.registrationRequired  =", spark.conf.get("spark.kryo.registrationRequired", "<unset>"))

**What you just saw.** Java's default serializer is portable but **bulky and slow**; Kryo
(`KryoSerializer`) is **smaller and faster** — a real win for shuffle- and cache-heavy jobs. Catch:
it's a **cluster-launch** setting (`--conf spark.serializer=...` at submit), **not** a reliable
mid-session flip from Connect, so we **read** it. (Tungsten columnar encoding sidesteps this for
built-in types; serialization bites hardest with RDDs / arbitrary objects — which you also can't
reach over Connect.)

## (5) Speculative execution — relaunch stragglers (but it can't fix skew)  ⚠️ *conf yes; relaunch is cluster-side*

In [ ]:
for k in ["spark.speculation", "spark.speculation.multiplier", "spark.speculation.quantile"]:
    print(f"{k:<32} = {spark.conf.get(k, '<unset / default>')}")

**What you just saw.** With `spark.speculation=true`, Spark relaunches a **straggler** task (far
slower than its peers, per `multiplier`/`quantile`) on another executor and keeps whichever finishes
first — great for **heterogeneous or flaky nodes** (a slow disk, a noisy neighbour). **The senior
catch: it cannot fix data skew (SPK-1).** A speculative copy of the hot-key task gets the *same*
giant partition — equally slow, just done twice. Speculation fixes *unlucky placement*, not
*unbalanced data*.

## (6) Broadcast — *variable* (JVM-only) vs *join* (the DataFrame analogue)  ✅/❌

Two things share the word "broadcast". A low-level **broadcast *variable*** (`sc.broadcast(x)`) ships
a read-only value to every executor once (not per task) — classic for a lookup map in an RDD
closure. It lives on `sparkContext`, so it's **JVM-only and raises on Connect**; we *describe* it.
The DataFrame-native analogue for the "small lookup table" case is a **broadcast join**,
`df.join(F.broadcast(dim), …)` — which we *can* show in a plan.

In [ ]:
# JVM-only — what a broadcast VARIABLE looks like. Would raise over Connect, so kept as a string.
print("JVM-only (NOT run over Connect):  bc = spark.sparkContext.broadcast({0: 'house'})\n")

# DataFrame analogue: a broadcast JOIN. `F.broadcast(dim)` hints Spark to ship the small side to
# every task -> no shuffle of the big fact. Watch the operator name in the plan.
bcast_join = fact.join(F.broadcast(dim), on="customer_id", how="inner")
bcast_join.explain(mode="formatted")

**What you just saw.** The plan shows a **`BroadcastHashJoin`** (with a `BroadcastExchange` feeding
the small side) instead of a `SortMergeJoin` — the big fact is **never shuffled**. That's the same
mechanism **SPK-5** (join strategies) lives in, and one of **SPK-1**'s skew fixes (broadcast the
small side → skew becomes irrelevant). The broadcast *join* is observable over Connect (it's in the
plan); the broadcast *variable* is not (it needs `sparkContext`).

## Prove it (the one measurable item) — WholeStageCodegen on vs off

Most "proof" here is the printed plan. The one place a before/after helps is **codegen on vs off**:
toggle `spark.sql.codegen.wholeStage` and `compare()` two runs of the same chained aggregation. On
tiny laptop data the wall-clock delta is **small and noisy** (codegen's win scales with rows &
CPU-bound work) — read the number as *directional*; the real takeaway is the **plan difference**.

In [ ]:
def run_chained():
    return (fact
        .filter(F.col("amount") > 50.0)
        .withColumn("amount_2x", F.col("amount") * 2)
        .groupBy("customer_id")
        .agg(F.sum("amount_2x").alias("total"))
        .count())

spark.conf.set("spark.sql.codegen.wholeStage", "true")
m_on = measure(spark, "codegen ON", run_chained)

spark.conf.set("spark.sql.codegen.wholeStage", "false")
m_off = measure(spark, "codegen OFF", run_chained)

spark.conf.set("spark.sql.codegen.wholeStage", "true")   # restore default
compare([m_on, m_off])
print("\n(Directional on tiny data — codegen's real win scales with rows & CPU-bound work.)")

## Takeaways & "in real production…"

- **Two of these you can *see* from a Connect notebook** — Catalyst plans, Tungsten codegen, and the
  broadcast-*join* plan all live in the **query plan**, so `df.explain(...)` is your window.
- **The rest you *tune* but don't observe from Connect** — unified-memory split, Kryo, speculation,
  broadcast *variables* are JVM-/launch-/cluster-side. Knowing **which bucket** a knob is in is the
  senior skill: don't flip a launch setting at runtime; don't expect Connect to expose a live
  executor-memory number.
- **`explain()` is the cheapest, most underused tool in Spark** — `extended=True` for what the
  optimizer did, `"codegen"` / `mode="formatted"` for what Tungsten will run, the join operator name
  to confirm a broadcast happened (SPK-5).
- **Internals connect Phase 1:** unified memory → executor OOM (SPK-2) & cache thrash (SPK-8);
  Catalyst rules → why a `CAST`/UDF kills pruning (SPK-7); speculation vs skew → why retries don't
  save SPK-1; broadcast → the first skew/join fix (SPK-1, SPK-5).
- **In prod:** keep AQE on, set serialization at submit, enable speculation only where node
  *heterogeneity* (not skew) is the problem, and reach for `explain()` *before* more executors.

## Teardown

Nothing was written — every demo generated data lazily and only `count()`-ed or `explain()`-ed it,
so there are no tables or files to remove. We just restore the production-tuned safety nets.

In [ ]:
apply_profile(spark, "tuned")        # restore production-tuned safety nets
spark.catalog.clearCache()
print("Done. Profile reset to 'tuned'. No tables/files were created; `make clean` clears .tmp if needed.")